# Analytics Engineering — Group 09 · CrediTrust Capital
**Role:** Analytics Engineer (AE) · **Dataset:** UCI Statlog German Credit · **Course:** BCDA 5P126

**Goal:** turn the model's default probabilities into credit decisions, and choose the decision threshold that minimises business cost.

**Business rule (from the brief):** approving a defaulter (Type II error) costs **5×** rejecting a good applicant (Type I error).

**Conventions used everywhere in this notebook**
| Item | Meaning |
|---|---|
| `y_true = 1` | applicant DEFAULTED (raw dataset class 2 = bad) |
| `y_true = 0` | good applicant (raw class 1) |
| `p_default` | model probability of default = P(class 2) |
| `p_default >= threshold` | **REJECT / manual review**; otherwise **APPROVE** |

**Run order:** Run the cells top to bottom from the repo root (or from `notebooks/`). Nothing else is needed.

## 0. Setup

In [1]:
# --- Colab only: uncomment the next two lines the first time ---
# !git clone https://github.com/SSRajeshwari-13/ML-Final-Lab-Group-09.git
# %cd ML-Final-Lab-Group-09

import sys, subprocess, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT = Path.cwd()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
PROC = PROJECT / "data" / "processed"
sys.path.insert(0, str(PROJECT / "src"))

import analytics_engine as ae          # our AE module (src/analytics_engine.py)

cfg = ae.CostConfig()                  # c_fp = 1, c_fn = 5, LGD = 0.60
print("Project folder :", PROJECT)
print(f"Type I cost = {cfg.c_fp}, Type II cost = {cfg.c_fn}, LGD assumption = {cfg.lgd}")

Project folder : C:\Users\Admin\Downloads
Type I cost = 1.0, Type II cost = 5.0, LGD assumption = 0.6


## 1. Generate predictions
`src/export_predictions.py` creates two files from the Data Scientist's saved model:
* `oof_predictions.csv` — 5-fold **out-of-fold** predictions on the training data → used to **choose** the threshold
* `test_predictions.csv` — predictions on the held-out test data → used **once** to **report** results

*Why two files?* Choosing the threshold on the test set would be tuning on test data, which the project rules forbid.

In [2]:
r = subprocess.run([sys.executable, str(PROJECT / "src" / "export_predictions.py")],
                   capture_output=True, text=True, cwd=PROJECT)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr)
    raise RuntimeError("export_predictions.py failed - read the error above")


C:\Users\Admin\anaconda3\python.exe: can't open file 'C:\\Users\\Admin\\Downloads\\src\\export_predictions.py': [Errno 2] No such file or directory



RuntimeError: export_predictions.py failed - read the error above

## 2. Load and sanity-check the prediction files

In [ ]:
oof  = pd.read_csv(PROC / "oof_predictions.csv")
test = pd.read_csv(PROC / "test_predictions.csv")

for name, d in (("OOF (train, used to tune)", oof), ("TEST (used to report)", test)):
    assert set(d["y_true"].unique()) <= {0, 1}, "y_true must be 0/1"
    assert d["p_default"].between(0, 1).all(), "p_default must be in [0, 1]"
    print(f"{name:26s} rows={len(d):4d}  default rate={d['y_true'].mean():.1%}  "
          f"mean p_default={d['p_default'].mean():.3f}")
test.head()

## 3. Confusion cost-matrix model
Rows = what really happened, columns = our decision. Correct decisions cost 0.

The **cost-optimal threshold** comes from comparing the two expected costs for an applicant with default probability *p*:

* approve → expected cost `p × c_fn`
* reject → expected cost `(1 − p) × c_fp`

Reject when `(1 − p)·c_fp < p·c_fn`, i.e. `p > c_fp / (c_fp + c_fn) = 1/6 ≈ 0.167`.

In [ ]:
display(ae.cost_matrix(cfg))
print(f"Theoretical cost-optimal threshold = {cfg.theoretical_threshold:.3f}")

## 4. Reference strategies on the test set
Before tuning anything, how costly are the simple alternatives? These give the model something to be compared against.

In [ ]:
n = len(test)
strategies = {
    "Approve everyone (no model)": np.zeros(n, dtype=int),
    "Reject everyone":             np.ones(n, dtype=int),
    "Model, default threshold 0.50": (test["p_default"] >= 0.50).astype(int),
}
ref = pd.DataFrame({k: ae.cost_metrics(test["y_true"], v, cfg) for k, v in strategies.items()}).T
ref[["TP", "FP", "FN", "TN", "total_cost", "recall_default", "approval_rate", "cost_weighted_f1"]]

## 5. Threshold tuning — on out-of-fold data only
Every threshold from 0.05 to 0.95 is evaluated on the OOF predictions. The chosen threshold is the one with the **lowest total cost** (ties broken toward higher recall).

In [ ]:
sweep = ae.threshold_sweep(oof["y_true"], oof["p_default"], cfg)
t_star = ae.best_threshold(sweep)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(sweep["threshold"], sweep["total_cost"], lw=2, label="Total cost (OOF)")
ax.axvline(t_star, color="green", ls="--", label=f"Chosen threshold = {t_star:.2f}")
ax.axvline(cfg.theoretical_threshold, color="orange", ls=":", label=f"Theory 1/6 = {cfg.theoretical_threshold:.3f}")
ax.axvline(0.5, color="grey", ls=":", label="Default 0.50")
ax.set_xlabel("Decision threshold on P(default)")
ax.set_ylabel("Total cost (Type I = 1, Type II = 5)")
ax.set_title("Cost vs. decision threshold (tuned on out-of-fold data)")
ax.legend(); plt.tight_layout(); plt.show()

print(f"Chosen threshold: {t_star:.2f}")
sweep.loc[sweep["threshold"].between(t_star - 0.10, t_star + 0.10)]

## 6. Report on the test set — once
The threshold from step 5 is applied to the **untouched test set** and compared with the default 0.50 and with approving everyone.

In [ ]:
pred_star = (test["p_default"] >= t_star).astype(int)
res_star  = ae.cost_metrics(test["y_true"], pred_star, cfg)
res_050   = ae.cost_metrics(test["y_true"], (test["p_default"] >= 0.5).astype(int), cfg)

final_cmp = pd.DataFrame({
    f"Tuned threshold ({t_star:.2f})": res_star,
    "Default threshold (0.50)": res_050,
}).T[["TP", "FP", "FN", "TN", "total_cost", "cost_per_applicant",
      "precision_default", "recall_default", "cost_weighted_f1", "approval_rate"]]
display(final_cmp)

cm = pd.DataFrame(
    [[res_star["TN"], res_star["FP"]], [res_star["FN"], res_star["TP"]]],
    index=["Actual GOOD", "Actual DEFAULT"], columns=["Decision APPROVE", "Decision REJECT"])
print("Confusion matrix at the tuned threshold (test set):")
display(cm)

print(f"Cost saved vs. approving everyone: {res_star['cost_saved_vs_approve_all']:.0f} cost units")
print(f"Cost saved vs. default 0.50      : {res_050['total_cost'] - res_star['total_cost']:.0f} cost units")

## 7. Derived business metrics per applicant
Adds `decision`, `risk_band`, expected cost of each action, `monthly_burden`, and **Expected Loss = PD × LGD × credit amount**.
This is the file the BI Developer uses for the dashboard.

In [ ]:
scored = ae.add_derived_metrics(test, t_star, cfg)
kpis = ae.portfolio_summary(scored)

display(scored.head())
print(json.dumps(kpis, indent=2))
print()
print(scored["risk_band"].value_counts())

## 8. Save the Analytics Engineer deliverables

In [ ]:
sweep.to_csv(PROC / "threshold_sweep.csv", index=False)
scored.to_csv(PROC / "scored_applicants.csv", index=False)
ae.cost_matrix(cfg).to_csv(PROC / "cost_matrix.csv")
ae.export_threshold_sheet(
    sweep, cfg, str(PROC / "Decision_Threshold_Tuning_Sheet.xlsx"),
    note="Sweep computed on out-of-fold predictions only.")

summary = {
    "config": {"c_fp": cfg.c_fp, "c_fn": cfg.c_fn, "lgd": cfg.lgd},
    "theoretical_threshold": round(cfg.theoretical_threshold, 4),
    "chosen_threshold": t_star,
    "test_results_tuned": res_star,
    "test_results_default_0.5": res_050,
    "portfolio_kpis": kpis,
}
with open(PROC / "ae_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

for f in ["threshold_sweep.csv", "scored_applicants.csv", "cost_matrix.csv",
          "Decision_Threshold_Tuning_Sheet.xlsx", "ae_summary.json"]:
    print(f"{f:40s}", "OK" if (PROC / f).exists() else "MISSING")

## 9. Apply the threshold to new applicants (demo of the live pipeline)
`predict.py` (ML Engineer) writes `prediction_results.csv` for new applicants. Here the tuned threshold turns those probabilities into decisions.

In [ ]:
new = pd.read_csv(PROC / "prediction_results.csv")
new["default_probability"] = new["Probability_Class_2"]      # class 2 = bad credit
new["final_decision"] = np.where(new["default_probability"] >= t_star,
                                 "Reject / Manual Review", "Approve")
new.to_csv(PROC / "ae_final_decisions.csv", index=False)
new[["Credit_Amount", "default_probability", "Predicted_Class", "final_decision"]]

## 10. Conclusion (numbers filled in automatically)

In [ ]:
print(f'''
ANALYTICS ENGINEERING CONCLUSION
--------------------------------
* Cost rule: Type II error (approve a defaulter) = {cfg.c_fn:g}x Type I error (reject a good applicant).
* Theory says reject when P(default) > {cfg.theoretical_threshold:.3f}. The out-of-fold sweep selected {t_star:.2f}.
* On the untouched test set ({res_star['n']} applicants):
    - tuned threshold {t_star:.2f}: total cost {res_star['total_cost']:.0f}, recall {res_star['recall_default']:.1%}, approval rate {res_star['approval_rate']:.1%}
    - default threshold 0.50 : total cost {res_050['total_cost']:.0f}, recall {res_050['recall_default']:.1%}, approval rate {res_050['approval_rate']:.1%}
    - approve everyone       : total cost {res_star['cost_approve_all']:.0f}
* Trade-off: a lower threshold catches more defaulters but rejects more good customers.
* Limitations: the test set is small ({res_star['n']} rows), LGD = {cfg.lgd:.0%} is an assumption, and the costs are unit costs from the brief, not real bank figures.
''')